In [5]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[0]))

import scripts.prepare as prepare
import scripts.services as services
import scripts.roi as roi_services
import scripts.bss_pipeline as bss_pl
import scripts.visualization as vis

from scipy.interpolate import make_interp_spline

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.signal import find_peaks
from scipy.signal import lfilter
from scipy.linalg import eig
from sklearn.decomposition import PCA

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
N_PC = 10 # discovered by analysis
NFRAMES_OUT = 800
SCALE_OUT = 0.2
FPS_OUT = 240
N_PEAKS = 5

VIDEO_PATH_IN = str(Path.cwd().parents[0] / "videos/camp_2/20260520/20260520_1/m/VID_20260520_072412821.mp4")
VIDEO_NAME = VIDEO_PATH_IN.split("/")[-1]
VIDEO_PATH_OUT = str(Path.cwd().parents[0] / "outputs/misc" / VIDEO_NAME)

ROIS = roi_services.load_rois(str(Path.cwd().parents[0] / "rois/rois.json"))
VIDEO_ROI = ROIS[VIDEO_NAME]

video_in_info = services.get_video_info(VIDEO_PATH_IN)
video_in_info

{'path': '/home/reginaldo/workspaces/PythonLanguage/marajo/videos/camp_2/20260520/20260520_1/m/VID_20260520_072412821.mp4',
 'fps': 29.923232187827843,
 'width': 1080,
 'height': 1920,
 'frames': 2639,
 'duration': 88.19,
 'shape': (1920, 1080, 3)}

In [ ]:
prepare.pre_processing(VIDEO_PATH_IN, VIDEO_PATH_OUT, NFRAMES_OUT, FPS_OUT, SCALE_OUT, VIDEO_ROI)
video_out_info = services.get_video_info(VIDEO_PATH_OUT)
video_out_info

In [3]:
def CP_alg(mixtures):

    n = 10

    ###################################
    # COMPUTE V AND U
    ###################################

    # Short and long half-lives
    shf = 1
    lhf = 900000

    # Max mask length
    max_mask_len = 50

    ###################################
    # Short-term mask
    ###################################

    h = shf
    t = int(n * h)

    lam = 2 ** (-1 / h)

    temp = np.arange(0, t)

    mask = lam ** temp
    mask[0] = 0
    mask = mask / np.sum(np.abs(mask))
    mask[0] = -1

    s_mask = mask

    ###################################
    # Long-term mask
    ###################################

    h = lhf
    t = int(n * h)
    t = min(t, max_mask_len)
    t = max(t, 1)

    lam = 2 ** (-1 / h)

    temp = np.arange(0, t)

    mask = lam ** temp
    mask[0] = 0
    mask = mask / np.sum(np.abs(mask))
    mask[0] = -1

    l_mask = mask

    ###################################
    # Filter each column of mixtures
    ###################################

    S = lfilter(s_mask, 1, mixtures, axis=0)
    L = lfilter(l_mask, 1, mixtures, axis=0)

    ###################################
    # Covariance matrices
    ###################################

    U = np.cov(S, rowvar=False, bias=True)
    V = np.cov(L, rowvar=False, bias=True)

    ###################################
    # Generalized eigenvalue problem
    ###################################

    eigvals, W = eig(V, U)

    W = np.real(W)

    ###################################
    # Extract sources
    ###################################

    ys = -(mixtures @ W)

    return ys, W

In [8]:
video_file = Path(VIDEO_PATH_OUT)
cap = cv.VideoCapture(str(video_file))
frames = []

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # gray
    if frame.ndim == 3:
        frame = frame[:, :, 0]

    frames.append(frame.reshape(-1))

cap.release()

dataset = np.asarray(frames, dtype=np.float32)


X = dataset.copy()
X -= X.mean(axis=0, keepdims=True)
pca = PCA()

W = pca.fit_transform(X.T)
H = pca.components_.T
V = pca.explained_variance_


mixtures = H[:, :N_PC]
unmixed, Wmix = CP_alg(mixtures)

Winvmix = np.fliplr(np.linalg.inv(Wmix))
unmixed = -np.fliplr(unmixed)

In [12]:
import numpy as np


def compute_source_spatial_maps(W, mixtures, unmixed, n_pc):
    """
    Calcula o mapa espacial associado a cada fonte separada.

    Parameters
    ----------
    W : np.ndarray
        Scores do PCA.
        Shape: (n_pixels, n_components)

    mixtures : np.ndarray
        Componentes principais fornecidos ao BSS.
        Shape: (n_frames, n_pc)

    unmixed : np.ndarray
        Fontes separadas pelo BSS.
        Shape: (n_frames, n_pc)

    n_pc : int
        Número de componentes principais utilizados no BSS.

    Returns
    -------
    spatial_maps : np.ndarray
        Peso de cada fonte em cada pixel.
        Shape: (n_pixels, n_pc)

    A : np.ndarray
        Matriz que reconstrói as misturas a partir das fontes.
        mixtures ~= unmixed @ A
    """

    W_selected = W[:, :n_pc]

    # Resolve:
    #
    # mixtures ≈ unmixed @ A
    #
    A, _, _, _ = np.linalg.lstsq(
        unmixed,
        mixtures,
        rcond=None,
    )

    # X.T ≈ W_selected @ mixtures.T
    #
    # mixtures ≈ unmixed @ A
    #
    # X.T ≈ W_selected @ A.T @ unmixed.T
    #
    spatial_maps = W_selected @ A.T

    return spatial_maps, A

In [13]:
spatial_maps, A = compute_source_spatial_maps(W, mixtures, unmixed, N_PC)

In [20]:
def create_source_overlay_video(input_video, output_video, spatial_map, alpha=0.45):
    cap = cv.VideoCapture(str(input_video))

    fps = cap.get(cv.CAP_PROP_FPS)
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv.VideoWriter_fourcc(*"mp4v")

    writer = cv.VideoWriter(str(output_video), fourcc, fps, (width, height))

    source_map = spatial_map.reshape(height, width)

    source_map = np.abs(source_map)
    source_map /= source_map.max() + 1e-12

    heatmap_uint8 = (source_map * 255).astype(np.uint8)

    heatmap = cv.applyColorMap(heatmap_uint8, cv.COLORMAP_JET)

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        overlay = cv.addWeighted(frame, 1.0 - alpha, heatmap, alpha, 0)

        writer.write(overlay)

    cap.release()
    writer.release()

In [21]:
create_source_overlay_video(VIDEO_PATH_OUT, "source_01_overlay.mp4", spatial_maps[:, 0])

In [22]:
for source_idx in range(N_PC):
    create_source_overlay_video(VIDEO_PATH_OUT, f"source_{source_idx + 1:02d}_overlay.mp4", spatial_maps[:, source_idx])